# Convert the CSV Data to RDF

This notebook is used to convert the CSV data into RDF format. The CSV files are read and processed, and the resulting RDF data is saved in Turtle format. The code is intentionally very verbose and not optimized for performance, as the main goal is to have a clear and understandable code that can be easily modified and extended.

## Create JSON

In the following part, all information from the different CSV files is combined into a **single deeply nested JSON** structure. This is done by iterating over the rows of the `party_csv` (which is the "main" file) and fetching all the related information from the other connected CSV files.

### Read CSV Files

In [1]:
from pathlib import Path
import pandas as pd

# path to all the CSV files relative to the location of this notebook
CSV_PATH = Path("../data/input")

# main file
party_csv = pd.read_csv(CSV_PATH / "party.csv")

# helper files
party_name_csv = pd.read_csv(CSV_PATH / "party_name.csv")
relationship_csv = pd.read_csv(CSV_PATH / "relationship.csv")
relationship_type_csv = pd.read_csv(CSV_PATH / "relationship_type.csv")
party_code_csv = pd.read_csv(CSV_PATH / "party_code.csv")
party_code_type_csv = pd.read_csv(CSV_PATH / "party_code_type.csv")

In [14]:
display(party_name_csv)

,party_name_id,party_id,notes,sources,translations_languages_code1,translations_abbreviation1,translations_name1,translations_internal_label1,translations_alternative_name1,translations_notes1,...,to_date_datetime,to_date_notes,to_date_sources,translations_languages_code4,translations_abbreviation4,translations_name4,translations_internal_label4,translations_alternative_name4,translations_notes4,translations_sources4
0,1,1,NaN,page_number_string: \r\nnotes: \r\nsource_full...,de-CH,SVP,Schweizerische Volkspartei,SVP,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,2,NaN,page_number_string: \r\nnotes: \r\nsource_full...,de-CH,SP Schweiz,Sozialdemokratische Partei der Schweiz,SP,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,3,NaN,page_number_string: 5\r\nnotes: \r\nsource_ful...,de-CH,NaN,FDP.Die Liberalen,FDPLIB,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,4,NaN,page_number_string: \r\nnotes: \r\nsource_full...,de-CH,Mitte,Die Mitte,Mitte,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,6,NaN,page_number_string: \r\nnotes: \r\nsource_full...,de-CH,GLP,Grünliberale Partei Schweiz,GLP,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
490,497,311,NaN,page_number_string: \r\nnotes: \r\nsource_full...,de-CH,NaN,Demokratische Partei Graubündens,Dem. GR,NaN,NaN,...,1971-06-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
491,498,311,NaN,page_number_string: \r\nnotes: \r\nsource_full...,de-CH,SVP Graubünden,Schweizerische Volkspartei Graubünden,SVP GR,NaN,NaN,...,2008-06-15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
492,499,311,NaN,page_number_string: \r\nnotes: \r\nsource_full...,de-CH,BPS Graubünden,Bürgerliche Partei Schweiz Graubünden,BPS GR,NaN,NaN,...,2008-07-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
493,500,311,NaN,page_number_string: \r\nnotes: \r\nsource_full...,de-CH,BDP Graubünden,Bürgerlich - Demokratische Partei Kanton Grau...,BDP GR,NaN,NaN,...,2021-06-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Date Handling

All the dates in the CSV and after import to a pandas DataFrame are given as strings in the form of "YYYY-MM-DDT12:00:00" format (always with the time set to 12:00:00). The time part is not relevant for our use case, so we will remove it. Furthermore, the end dates are set to the date of the last valid day, so that schema:validThrough can be used:

- `schema:validFrom` means the first valid day
- `schema:validThrough` means the last valid day

Unknown dates are represented as `NaN` in the DataFrame. To check for unknown dates, we will use the `pd.isna()` function instead of checking for `None`, as the latter does not work with `NaN` values.

In [2]:
def convert_time_column(df, column_name, end_date=False):
    """
    Convert the time column to a date format and remove the time part. 
    If it is an end date, subtract one day to get the last valid day.
    Convert it back to strings in the format YYYY-MM-DD.
    """
    df[column_name] = pd.to_datetime(df[column_name], errors='coerce').dt.date
    if end_date:
        df[column_name] = df[column_name] - pd.Timedelta(days=1)
    df[column_name] = df[column_name].astype(str)
    return df

In [3]:
# convert the time columns to date format and remove the time part

party_csv = convert_time_column(party_csv, 'founding_date_datetime')
party_csv = convert_time_column(party_csv, 'dissolution_date_datetime', end_date=True)
party_name_csv = convert_time_column(party_name_csv, 'from_date_datetime')
party_name_csv = convert_time_column(party_name_csv, 'to_date_datetime', end_date=True)
party_relationship_csv = convert_time_column(relationship_csv, 'from_date_datetime')
party_relationship_csv = convert_time_column(relationship_csv, 'to_date_datetime', end_date=True)

### Create Parties JSON

In [4]:
def parse_list(s):
    """
    Parse a string representation of a list like "[1, 2, 3]" into a Python list.
    """
    if pd.isna(s) or s == "[]":
        return []
    return [int(item) for item in s.strip("[]").split(",")]

In [ ]:
# create the main dictionary that will hold all the parties and their information
parties = {}

# for each identity (each row in the party_csv)
for _, row in party_csv.iterrows():
    
    # Create a dictionary for the current party
    party = {}
    
    # get the party_id - needed to set the key in the parties dictionary
    party_id = row["party_id"]

    # basic date information
    party["founding_date"] = row["founding_date_datetime"]
    party["dissolution_date"] = row["dissolution_date_datetime"]

    
    #### names ####
    
    # get party names
    party_name_ids = parse_list(row["party_name_id"])

    # all names are stored in a list of dictionaries
    names = []

    for party_name_id in party_name_ids:
        
        # names
        name_lang = {}
        for lang_id in range(1, 5):  # there are 4 language columns (1, 2, 3, 4)
            lang_name = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, f"translations_languages_code{lang_id}"].item()
            name = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, f"translations_name{lang_id}"].item()
            if str(lang_name) != "nan" and str(name) != "nan":
                name_lang[str(lang_name)[0:2]] = name
        
        # abbreviations
        abbr_lang = {}
        for lang_id in range(1, 5):  # there are 4 language columns (1, 2, 3, 4)
            lang_abbr = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, f"translations_languages_code{lang_id}"].item()
            abbr = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, f"translations_abbreviation{lang_id}"].item()
            if str(lang_abbr) != "nan" and str(abbr) != "nan":
                abbr_lang[str(lang_abbr)[0:2]] = abbr
        
        # dates for the name and abbreviation
        valid_from = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, "from_date_datetime"].item()
        valid_through = party_name_csv.loc[party_name_csv["party_name_id"] == party_name_id, "to_date_datetime"].item()
        
        # create a name entry and add it to the names list
        name_entry = {"id": party_name_id, "name": name_lang, "abbr": abbr_lang, "valid_from": valid_from, "valid_through": valid_through}
        names.append(name_entry)

    
    # sort names by from_date
    names.sort(key=lambda x: x["valid_from"])
    
    # add names to the party dictionary
    party["names"] = names

    
    #### relationships ####

    # get relationships, they are given in two columns, relationship_id_1 and relationship_id_2.
    # They go to either shape-changing or hierarchical relationships.
    relationships = []

    shape_changing_relationship_types = ["successor-of", "split-from", "accessed"]
    hierarchical_relationship_types = ["child-of", "observer-in", "affiliated-with"]

    # get relationships from both columns and combine them into a single list
    relationship_ids = parse_list(row["relationship_id_1"]) + parse_list(row["relationship_id_2"])

    for relationship_id in relationship_ids:
        
        relationship_from = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "party_1_id"].item()
        relationship_to = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "party_2_id"].item()
        relationship_type = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "type"].item()
        relationship_label = relationship_type_csv.loc[relationship_type_csv["relationship_type_id"] == relationship_type, "identifier"].item()
        valid_from = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "from_date_datetime"].item()
        valid_through = relationship_csv.loc[relationship_csv["relationship_id"] == relationship_id, "to_date_datetime"].item()
        
        relationship_entry = {"id": relationship_id, "type_label": relationship_label,"party_from": relationship_from, "party_to": relationship_to, "valid_from": valid_from, "valid_through": valid_through}
        
        if relationship_entry["type_label"] in shape_changing_relationship_types:
            relationship_entry["supra_type_label"] = "shape-changing"
            relationships.append(relationship_entry)
        else:
            # hierarchical relationships are only added on one side because of "version-avalanche" effect.
            if relationship_entry["party_to"] != party_id:
                relationship_entry["supra_type_label"] = "hierarchical"
                relationships.append(relationship_entry)

    # sort relationships by from_date and add them to the party dictionary
    relationships.sort(key=lambda x: x["valid_from"])
    
    # add relationships to the party dictionary
    party["relationships"] = relationships

    
    #### party identifiers ####

    # different identifiers for the same party (e.g. bfs-code, wikidata, etc.)
    party_code_ids = parse_list(row["party_code_id"])
    party_codes = []

    for party_code_id in party_code_ids:
        code = party_code_csv.loc[party_code_csv["party_code_id"] == party_code_id, "code"].item()
        code_type = party_code_csv.loc[party_code_csv["party_code_id"] == party_code_id, "party_code_type_id"].item()
        code_type_label = party_code_type_csv.loc[party_code_type_csv["party_code_type_id"] == code_type, "identifier"].item()

        code_entry = {"id": party_code_id, "code": code, "type_label": code_type_label}
        party_codes.append(code_entry)

    party_codes.append({"code": row["identifier"], "type_label": "ap-identifier"})  # add the identifier from the party.csv row

    #party["party_codes"] = party_codes

    
    #### versions ####

    # versions will be created either through a name change or through a relationship change. If both happen on the same date, only one version will be created.

    # get all unique version ids from names and relationships, 
    # they are given by the valid_from of the name or relationship
    version_ids = set()
    
    for name in names:
        version_ids.add(name["valid_from"])
    for relationship in relationships:
        version_ids.add(relationship["valid_from"])

    version_ids = sorted(list(version_ids))

    versions = {}

    for index, version_id in enumerate(version_ids):

        if party_id == 43 and version_id == "1987-07-01":
            breakpoint()
        
        # build empty version dictionary
        version = {
            "valid_from": None, 
            "valid_through": None,
            "is_founding": False,
            "is_dissolution": False,
            "is_name_change": False,
            "name": None, 
            "relationships": []
        }
        
        # version_from and version_through are also used later on
        version_from = version_id
        version["valid_from"] = version_from
        
        version_through = None
        # get valid_through from the next version_id, or None if it's the last version
        if version_ids.index(version_id) < len(version_ids) - 1:
            version_through = version_ids[version_ids.index(version_id) + 1]
            # remove one day from the valid_through date to get the last valid day
            version_through = pd.to_datetime(version_through) - pd.Timedelta(days=1)
            version_through = str(version_through.date())
            version["valid_through"] = version_through

        # first version is founding version
        if index == 0:
            version["is_founding"] = True

        # last version is dissolution version if the party has a dissolution date
        if index == len(version_ids) - 1 and party["dissolution_date"] is not None:
            version["is_dissolution"] = True
            version["valid_through"] = party["dissolution_date"]

        # get currently valid name for this version
        version_name = None
        for name_entry in names:
            
            name_from = name_entry["valid_from"]
            name_through = name_entry["valid_through"]

            if name_from == version_from and index != 0: # first version is founding version, so no name change can happen
                version["is_name_change"] = True

            # for possibilities of _through dates being None
            if pd.isna(version_through) and pd.isna(name_through):
                if name_from <= version_from:
                    version_name = name_entry
            elif pd.isna(version_through) and not pd.isna(name_through): # cannot happen
                continue
            elif not pd.isna(version_through) and pd.isna(name_through):
                if name_from <= version_from:
                    version_name = name_entry
            elif not pd.isna(version_through) and not pd.isna(name_through):
                if name_from <= version_from and version_from <= name_through:
                    version_name = name_entry

        version["name"] = version_name

        # get currently valid relationships for this version
        version_relationships = []
        for relationship_entry in relationships:
            
            if relationship_entry["supra_type_label"] == "shape-changing":
                if relationship_entry["valid_from"] == version_id:
                    version_relationships.append(relationship_entry)
            
            elif relationship_entry["supra_type_label"] == "hierarchical":
                
                relationship_from = relationship_entry["valid_from"]
                relationship_through = relationship_entry["valid_through"]

                if pd.isna(version_through) and pd.isna(relationship_through):
                    if relationship_from <= version_from:
                        version_relationships.append(relationship_entry)
                elif pd.isna(version_through) and not pd.isna(relationship_through): # cannot happen
                    continue
                elif not pd.isna(version_through) and pd.isna(relationship_through):
                    if relationship_from <= version_from:
                        version_relationships.append(relationship_entry)
                elif not pd.isna(version_through) and not pd.isna(relationship_through):
                    if relationship_from <= version_from and version_from <= relationship_through:
                        version_relationships.append(relationship_entry)

        version["relationships"] = version_relationships

        # add version to versions dictionary
        versions[version_id] = version

    # sort versions by key
    versions = dict(sorted(versions.items(), key=lambda x: x[0]))

    # check if the last version has the same valid_from and valid_through date and is_dissolution == True,
    # if so, add is_dissolution = True and the relationships to the previous version and remove the last version

    # example for this is party 9
    if len(versions) > 1:
        last_version_id = list(versions.keys())[-1]
        last_version = versions[last_version_id]
        second_last_version_id = list(versions.keys())[-2]
        second_last_version = versions[second_last_version_id]

        if not pd.isna(last_version["valid_through"]):
            if last_version["valid_from"] >= last_version["valid_through"] and last_version["is_dissolution"]:
                second_last_version["is_dissolution"] = True
                second_last_version["relationships"].extend(last_version["relationships"])
                del versions[last_version_id]

    party["versions"] = versions


    # add party to parties dictionary (finish line operations)
    parties[party_id] = party

In [15]:
import json

print(json.dumps(parties, indent=2, ensure_ascii=False))

{
  "1": {
    "founding_date": "1971-09-22",
    "dissolution_date": NaN,
    "names": [
      {
        "id": 1,
        "name": {
          "de": "Schweizerische Volkspartei",
          "fr": "Union Démocratique du Centre",
          "en": "Schweizerische Volkspartei"
        },
        "abbr": {
          "de": "SVP",
          "fr": "UDC"
        },
        "valid_from": "1971-09-22",
        "valid_through": NaN
      }
    ],
    "relationships": [
      {
        "id": 3,
        "type_label": "successor-of",
        "party_from": 1,
        "party_to": 11,
        "valid_from": "1971-09-22",
        "valid_through": NaN,
        "supra_type_label": "shape-changing"
      },
      {
        "id": 9,
        "type_label": "successor-of",
        "party_from": 1,
        "party_to": 18,
        "valid_from": "1971-09-22",
        "valid_through": NaN,
        "supra_type_label": "shape-changing"
      },
      {
        "id": 5,
        "type_label": "split-from",
        "party_

## Convert the Deep JSON to RDF

In [7]:
def get_last_version_before_date(chronology, date):
    last_version = None
    for d in sorted(chronology.keys()):
        if d < date:
            last_version = d
        else:
            break
    return last_version

In [8]:
from datetime import datetime

def get_all_parent_versions(parent_versions, date_from, date_through, version_from):
    
    # if no date_through is given, use today as the end date
    if pd.isna(date_through):
        date_through = str(datetime.now().strftime("%Y-%m-%d"))
    date_from = str(date_from)[0:10]
    date_through = str(date_through)[0:10]
    
    applicable_parent_versions = []
    
    for entry in parent_versions:
        entry_from = parent_versions[entry]["valid_from"]
        entry_through = parent_versions[entry]["valid_through"]
        if pd.isna(entry_through):
            entry_through = str(datetime.now().strftime("%Y-%m-%d"))
        
        if (entry_from <= date_from and date_from <= entry_through) or (entry_from <= date_through and date_through <= entry_through):
            
            if entry_from <= version_from:
                applicable_parent_versions.append(entry_from)

    return applicable_parent_versions

### URIs

In [ ]:
PARTY_IDENTITY = "https://politics.ld.admin.ch/party-id"
PARTY_VERSION = "https://politics.ld.admin.ch/party-version"
PARTY_VOCAB = "https://politics.schema.ld.admin.ch/party-vocab"
PARTY_PREFIX = "pol"

### Transformation

In [10]:
from rdflib import Graph, Literal, RDF, URIRef, Namespace

# create graph without binding namespaces to avoid automatic prefixing
g = Graph(bind_namespaces="none")

schema = Namespace("http://schema.org/")
g.bind("schema", schema)
xsd = Namespace("http://www.w3.org/2001/XMLSchema#")
g.bind("xsd", xsd)
vl = Namespace("https://version.link/")
g.bind("vl", vl)
pol = Namespace(PARTY_VOCAB)
g.bind(PARTY_PREFIX, pol)

for party_id, party in parties.items():
    
    # create identity
    identity_uri = URIRef(f"{PARTY_IDENTITY}/{party_id}")
    g.add((identity_uri, RDF.type, vl.Identity))
    if not pd.isna(party["dissolution_date"]):
        g.add((identity_uri, RDF.type, vl.Deprecated))

    
    # create versions
    version_ids = list(party["versions"].keys())

    for i in range(len(version_ids)):
        
        version_id = version_ids[i]
            
        version_uri = URIRef(f"{PARTY_VERSION}/{party_id}_{version_id}")
        g.add((version_uri, RDF.type, vl.Version))
        g.add((version_uri, vl.identity, identity_uri))

        # add dates
        g.add((version_uri, schema.validFrom, Literal(party["versions"][version_id]["valid_from"], datatype=xsd.date)))
        if not pd.isna(party["versions"][version_id]["valid_through"]):
            g.add((version_uri, schema.validThrough, Literal(party["versions"][version_id]["valid_through"], datatype=xsd.date)))

        # add name
        name_entry = party["versions"][version_id]["name"]
        if not pd.isna(name_entry):
            for lang, name in name_entry["name"].items():
                g.add((version_uri, schema.name, Literal(name, lang=lang)))
            for lang, abbr in name_entry["abbr"].items():
                g.add((version_uri, schema.alternateName, Literal(abbr, lang=lang)))

        # link to successors and predecessors
        # only link to successor if the current is not the last versions entry 
        if i < len(version_ids) - 1:
            next_version_id = version_ids[i + 1]
            next_version_uri = URIRef(f"{PARTY_VERSION}/{party_id}_{next_version_id}")
            g.add((version_uri, vl.successor, next_version_uri))
            g.add((next_version_uri, vl.predecessor, version_uri))

        # if it is last entry
        if i == len(version_ids) - 1:
            g.add((identity_uri, vl.version, version_uri))


        # check all relationships for special cases
        for relationship in party["versions"][version_id].get("relationships", []):
            
            # special case split_from, successor_of, accessed are such, that they create new versions for all parties involved
            # special case child-of, observer-in, affiliated-with are more "hierarchical" and do not create new versions on the other side of the relationship (e.g. child-of, observer-in, affiliated-with)

            # special case 1: split-from
            if relationship["type_label"] == "split-from" and relationship["party_from"] == party_id:
                parent_party_id = relationship["party_to"]
                parent_version_id = get_last_version_before_date(parties[parent_party_id]["versions"], version_id)
                if not pd.isna(parent_version_id):
                    parent_version_uri = URIRef(f"{PARTY_VERSION}/{parent_party_id}_{parent_version_id}")
                    g.add((parent_version_uri, vl.successor, version_uri))
                    g.add((parent_version_uri, pol.splitTo, version_uri))
                    g.add((version_uri, vl.predecessor, parent_version_uri))
                    g.add((version_uri, pol.splitFrom, parent_version_uri))

            # special case 2: successor-of
            if relationship["type_label"] == "successor-of" and relationship["party_from"] == party_id:
                parent_party_id = relationship["party_to"]
                parent_version_id = get_last_version_before_date(parties[parent_party_id]["versions"], version_id)
                if not pd.isna(parent_version_id):
                    parent_version_uri = URIRef(f"{PARTY_VERSION}/{parent_party_id}_{parent_version_id}")
                    g.add((parent_version_uri, vl.successor, version_uri))
                    g.add((parent_version_uri, pol.successor, version_uri))
                    g.add((version_uri, vl.predecessor, parent_version_uri))
                    g.add((version_uri, pol.predecessor, parent_version_uri))

            # special case 3: accessed
            if relationship["type_label"] == "accessed" and relationship["party_to"] == party_id:
                parent_party_id = relationship["party_from"]
                parent_version_id = get_last_version_before_date(parties[parent_party_id]["versions"], version_id)
                if not pd.isna(parent_version_id):
                    parent_version_uri = URIRef(f"{PARTY_VERSION}/{parent_party_id}_{parent_version_id}")
                    g.add((parent_version_uri, vl.successor, version_uri))
                    g.add((parent_version_uri, pol.accessed, version_uri))
                    g.add((version_uri, vl.predecessor, parent_version_uri))
                    g.add((version_uri, pol.accessedBy, parent_version_uri))

            # special case 4: child-of
            if relationship["type_label"] == "child-of":
                    parent_party_id = relationship["party_to"]
                    if party_id == 101 and version_id == "1979-03-05":
                        breakpoint()
                    parent_version_ids = get_all_parent_versions(parties[parent_party_id]["versions"], 
                                                                 relationship["valid_from"], 
                                                                 relationship["valid_through"],
                                                                 party["versions"][version_id]["valid_from"])
                    if parent_version_ids:
                        for parent_version_id in parent_version_ids:
                            parent_version_uri = URIRef(f"{PARTY_VERSION}/{parent_party_id}_{parent_version_id}")
                            g.add((version_uri, schema.isPartOf, parent_version_uri))
                            g.add((version_uri, pol.childOf, parent_version_uri))


g.serialize(destination="../data/output/parties.ttl", format="turtle")

<Graph identifier=N244e59a915d645d689ce4bd15750275f (<class 'rdflib.graph.Graph'>)>